In [0]:
import time
from datetime import datetime

dbutils.widgets.text("github_token", "")

dbutils.widgets.text("repos", "langchain-ai/langchain,run-llama/llama_index,chroma-core/chroma,qdrant/qdrant,deepset-ai/haystack")
dbutils.widgets.text("start_date", "2024-10-10")
dbutils.widgets.text("end_date", "2026-10-10")
dbutils.widgets.text("load_type", "full")
dbutils.widgets.text("endpoints", "commits,issues")
dbutils.widgets.text("max_pages", "0")          # 0 = pull everything

g = lambda k: dbutils.widgets.get(k)
common = {"load_type": g("load_type"), "start_date": g("start_date"), "end_date": g("end_date")}
failed = []

# 1) Source -> Raw, one repo at a time (so a failure only costs one repo)
for repo in [r.strip() for r in g("repos").split(",")]:
    t = time.time()
    try:
        dbutils.notebook.run("./01_ingest_raw", 0, {**common,
            "repos": repo, "github_token": g("github_token"),
            "endpoints": g("endpoints"), "max_pages": g("max_pages")})
        print(f"OK   {repo} ({int(time.time()-t)}s)")
    except Exception as e:
        failed.append(repo)
        print(f"FAIL {repo}: {str(e)[:200]}")

# 2) Raw -> Bronze (skips files already loaded)
dbutils.notebook.run("./02_raw_to_bronze", 0, {"load_type": g("load_type"), "ingest_date": "*"})

# 3) Bronze -> Silver (MERGE, safe to rerun)
dbutils.notebook.run("./03_bronze_to_silver", 0, {"load_type": g("load_type")})

# 4) Silver -> Gold (refreshes automatically after Silver)
dbutils.notebook.run("./05_silver_to_gold", 0, {"load_type": g("load_type"), "gold_mode": "incremental"})

print("Done. Failed repos:", failed or "none")
display(spark.sql("""SELECT layer, status, SUM(rows_inserted) AS inserted, SUM(rows_updated) AS updated
                     FROM workspace.rag.pipeline_execution_logs GROUP BY layer, status ORDER BY layer"""))

FAIL langchain-ai/langchain: An error occurred while calling NotebookRun.
: com.databricks.WorkflowException: com.databricks.NotebookExecutionException: FAILED: Workload failed, see run output for details
	at com.databricks.workf
FAIL run-llama/llama_index: An error occurred while calling NotebookRun.
: com.databricks.WorkflowException: com.databricks.NotebookExecutionException: FAILED: Workload failed, see run output for details
	at com.databricks.workf
FAIL chroma-core/chroma: An error occurred while calling NotebookRun.
: com.databricks.WorkflowException: com.databricks.NotebookExecutionException: FAILED: Workload failed, see run output for details
	at com.databricks.workf
FAIL qdrant/qdrant: An error occurred while calling NotebookRun.
: com.databricks.WorkflowException: com.databricks.NotebookExecutionException: FAILED: Workload failed, see run output for details
	at com.databricks.workf
FAIL deepset-ai/haystack: An error occurred while calling NotebookRun.
: com.databricks.Workfl

layer,status,inserted,updated
Bronze-to-Silver,Success,112734,59
Gold-Optimize,Success,0,0
Raw-to-Bronze,Success,110189,0
Raw-to-Bronze,Failure,0,0
Silver-to-Gold,Success,184275,0
Source-to-Raw,Failure,0,0
Source-to-Raw,Success,115189,0


In [0]:
display(spark.sql("""
SELECT layer, parameter_or_file, left(error_message, 300) AS error, start_time
FROM workspace.rag.pipeline_execution_logs
WHERE status = 'Failure' ORDER BY start_time DESC LIMIT 10"""))

display(spark.sql("""
SELECT parameter_or_file, rows_inserted, rows_updated, start_time, end_time
FROM workspace.rag.pipeline_execution_logs
WHERE layer = 'Silver-to-Gold' ORDER BY start_time DESC LIMIT 12"""))

for t in ["dim_repo","dim_date","dim_contributor","fact_commit_activity","fact_issue_lifecycle","agg_daily_repo_health"]:
    print(t, spark.table(t).count())

layer,parameter_or_file,error,start_time
Raw-to-Bronze,/Volumes/workspace/rag/raw/landing/incremental/issues/*/*/*.json,[PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/workspace/rag/raw/landing/incremental/issues/*/*/*.json. SQLSTATE: 42K03 JVM stacktrace: org.apache.spark.sql.AnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3556) at or,2026-10-25T15:57:47.956Z
Raw-to-Bronze,/Volumes/workspace/rag/raw/landing/incremental/commits/*/*/*.json,[PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/workspace/rag/raw/landing/incremental/commits/*/*/*.json. SQLSTATE: 42K03 JVM stacktrace: org.apache.spark.sql.AnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3556) at o,2026-10-25T15:57:31.759Z
Raw-to-Bronze,/Volumes/workspace/rag/raw/landing/incremental/issues/*/*/*.json,[PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/workspace/rag/raw/landing/incremental/issues/*/*/*.json. SQLSTATE: 42K03 JVM stacktrace: org.apache.spark.sql.AnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3556) at or,2026-10-25T08:15:55.837Z
Raw-to-Bronze,/Volumes/workspace/rag/raw/landing/incremental/commits/*/*/*.json,[PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/workspace/rag/raw/landing/incremental/commits/*/*/*.json. SQLSTATE: 42K03 JVM stacktrace: org.apache.spark.sql.AnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3556) at o,2026-10-25T08:15:52.456Z
Raw-to-Bronze,/Volumes/workspace/rag/raw/landing/incremental/issues/*/*/*.json,[PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/workspace/rag/raw/landing/incremental/issues/*/*/*.json. SQLSTATE: 42K03 JVM stacktrace: org.apache.spark.sql.AnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3556) at or,2026-10-25T07:59:44.535Z
Raw-to-Bronze,/Volumes/workspace/rag/raw/landing/incremental/commits/*/*/*.json,[PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/workspace/rag/raw/landing/incremental/commits/*/*/*.json. SQLSTATE: 42K03 JVM stacktrace: org.apache.spark.sql.AnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3556) at o,2026-10-25T07:59:25.755Z
Source-to-Raw,deepset-ai/haystack/issues,"Rate limit exhausted, resets in 2538s",2026-10-11T07:09:01.452Z
Source-to-Raw,deepset-ai/haystack/commits,"Rate limit exhausted, resets in 2540s",2026-10-11T07:08:58.595Z
Source-to-Raw,qdrant/qdrant/issues,"Rate limit exhausted, resets in 2560s",2026-10-11T07:08:36.437Z
Source-to-Raw,chroma-core/chroma/issues,"Rate limit exhausted, resets in 908s",2026-10-11T06:53:05.902Z


parameter_or_file,rows_inserted,rows_updated,start_time,end_time
"fact_issue_lifecycle (merge, mode=full, input=36927)",0,0,2026-10-25T15:57:39.745Z,2026-10-25T15:57:49.847Z
"fact_commit_activity (merge, mode=full, input=17459)",0,0,2026-10-25T15:57:22.128Z,2026-10-25T15:57:38.078Z
dim_contributor (overwrite),9442,0,2026-10-25T15:57:13.699Z,2026-10-25T15:57:20.151Z
dim_date (overwrite),2464,0,2026-10-25T15:57:06.158Z,2026-10-25T15:57:11.397Z
dim_repo (overwrite),5,0,2026-10-25T15:56:40.834Z,2026-10-25T15:56:58.331Z
agg_daily_repo_health (overwrite),4943,0,2026-10-25T08:17:35.840Z,2026-10-25T08:17:41.225Z
"fact_issue_lifecycle (merge, nothing new)",0,0,2026-10-25T08:17:33.680Z,2026-10-25T08:17:34.496Z
"fact_commit_activity (merge, nothing new)",0,0,2026-10-25T08:17:30.549Z,2026-10-25T08:17:31.422Z
dim_contributor (overwrite),9442,0,2026-10-25T08:17:23.927Z,2026-10-25T08:17:28.331Z
dim_date (overwrite),2464,0,2026-10-25T08:17:19.656Z,2026-10-25T08:17:22.150Z


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6782920620487192>, line 12
      6 display(spark.sql("""
      7 SELECT parameter_or_file, rows_inserted, rows_updated, start_time, end_time
      8 FROM workspace.rag.pipeline_execution_logs
      9 WHERE layer = 'Silver-to-Gold' ORDER BY start_time DESC LIMIT 12"""))
     11 for t in ["dim_repo","dim_date","dim_contributor","fact_commit_activity","fact_issue_lifecycle","agg_daily_repo_health"]:
---> 12     print(t, spark.table(t).count())

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:381, in DataFrame.count(self)
    380 def count(self) -> int:
--> 381     table, _ = self.agg(F._invoke_function("count", F.lit(1)))._to_table()  # type: ignore[operator]
    382     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2168,

In [0]:
%sql
SELECT layer, parameter_or_file, status, rows_inserted, left(error_message, 300) AS error, start_time
FROM workspace.rag.pipeline_execution_logs
WHERE load_type = 'incremental'
ORDER BY start_time DESC LIMIT 15

layer,parameter_or_file,status,rows_inserted,error,start_time
Gold-Optimize,"agg_daily_repo_health CLUSTER BY (repo_key, date_key)",Success,0,null,2026-10-25T16:25:13.582Z
Gold-Optimize,"fact_issue_lifecycle CLUSTER BY (repo_key, created_date_key)",Success,0,null,2026-10-25T16:25:06.524Z
Gold-Optimize,"fact_commit_activity CLUSTER BY (repo_key, date_key)",Success,0,null,2026-10-25T16:24:56.957Z
Silver-to-Gold,agg_daily_repo_health (overwrite),Success,4943,null,2026-10-25T16:24:41.782Z
Silver-to-Gold,"fact_issue_lifecycle (merge, nothing new)",Success,0,null,2026-10-25T16:24:39.313Z
Silver-to-Gold,"fact_commit_activity (merge, nothing new)",Success,0,null,2026-10-25T16:24:35.311Z
Silver-to-Gold,dim_contributor (overwrite),Success,9442,null,2026-10-25T16:24:27.791Z
Silver-to-Gold,dim_date (overwrite),Success,2464,null,2026-10-25T16:24:16.952Z
Silver-to-Gold,dim_repo (overwrite),Success,5,null,2026-10-25T16:24:09.964Z
Bronze-to-Silver,"silver_issue_labels <- ingest_date=*, bronze_batch_id=*",Success,0,null,2026-10-25T16:23:37.044Z
